# TRELLIS – Text/Image to 3D (Flash Attention 2 + احتياطي)
تشغيل [TRELLIS](https://github.com/JeffreyXiang/TRELLIS) على Google Colab.

**تحسينات رئيسية:**
- ✅ تثبيت `trellis` مباشرة من GitHub **بدون git clone** (لا مشاكل مصادقة).
- ✅ دعم Flash Attention 2 تلقائي عند توفره، وإلا `sdpa`.
- ✅ معالجة أخطاء شاملة في كل خطوة.

In [ ]:
import torch, subprocess, sys

print(f"PyTorch: {torch.__version__}")
if not torch.cuda.is_available():
    raise RuntimeError("الرجاء تفعيل GPU من Runtime → Change runtime type")

!nvidia-smi
gpu_name = subprocess.getoutput("nvidia-smi --query-gpu=name --format=csv,noheader")
print(f"🔹 GPU: {gpu_name}")

SUPPORTED_GPU = ["A100", "A6000", "3090", "4090", "L4", "L40", "H100"]
USE_FLASH_ATTN = any(g in gpu_name for g in SUPPORTED_GPU)
print("✨ يدعم Flash Attention 2" if USE_FLASH_ATTN else "ℹ️  يستخدم SDPA (انتباه PyTorch الافتراضي).")

In [ ]:
import pkg_resources

def is_installed(pkg_name):
    try:
        pkg_resources.get_distribution(pkg_name)
        return True
    except:
        return False

print("🔍 فحص الحزم الأساسية...")
base_pkgs = ["trimesh", "pygltflib", "viser", "omegaconf", "opencv-python", "imageio", "gradio", "huggingface_hub"]
for pkg in base_pkgs:
    if not is_installed(pkg):
        !pip install -q {pkg}
        print(f"✔️ {pkg}")
    else:
        print(f"✅ {pkg}")

# spconv
if not is_installed("spconv"):
    cuda_ver = torch.version.cuda.replace(".", "")
    try:
        !pip install -q spconv-cu{cuda_ver}
    except:
        !pip install -q spconv
    print("✔️ spconv")
else:
    print("✅ spconv")

# Flash Attention 2 (اختياري)
if USE_FLASH_ATTN and not is_installed("flash-attn"):
    print("⚡ تثبيت Flash Attention 2...")
    !pip install -q ninja packaging
    !pip install flash-attn --no-build-isolation
    import flash_attn
    print("✅ flash-attn")

print("\n✅ جميع الحزم الأساسية جاهزة.")

In [ ]:
import os, sys, subprocess

# الطريقة 1: محاولة التثبيت المباشر من GitHub عبر pip (لا تحتاج git clone)
print("⏳ تثبيت مكتبة trellis من GitHub...")
installed = False
try:
    !pip install git+https://github.com/JeffreyXiang/TRELLIS.git
    import trellis
    installed = True
    print("✅ تم تثبيت trellis بنجاح عبر pip.")
except Exception as e:
    print(f"⚠️ فشل التثبيت عبر pip: {e}")
    # الطريقة 2: تنزيل الأرشيف وفك الضغط يدوياً
    print("محاولة تنزيل المستودع كـ ZIP...")
    !wget -q https://github.com/JeffreyXiang/TRELLIS/archive/refs/heads/main.zip -O /tmp/trellis.zip
    !unzip -q /tmp/trellis.zip -d /tmp
    trellis_dir = "/tmp/TRELLIS-main"
    if os.path.isdir(trellis_dir):
        sys.path.insert(0, trellis_dir)
        import trellis
        installed = True
        print("✅ تم استخراج trellis من الأرشيف وإضافتها إلى المسار.")
    else:
        raise FileNotFoundError("لم يتم العثور على مجلد المستودع بعد فك الضغط.")

if not installed:
    raise RuntimeError("تعذر تثبيت trellis بأي طريقة.")

# تحديد وضع الانتباه المناسب
if USE_FLASH_ATTN:
    try:
        import flash_attn
        attn_impl = "flash_attention_2"
        print("🔧 Flash Attention 2 مفعل.")
    except:
        attn_impl = "sdpa"
        print("🔧 Flash Attention غير متاح، استخدام SDPA.")
else:
    attn_impl = "sdpa"
    print("🔧 SDPA مفعل.")

In [ ]:
import torch
from trellis.pipelines import TrellisImageTo3DPipeline

try:
    pipeline = TrellisImageTo3DPipeline.from_pretrained(
        "JeffreyXiang/TRELLIS-image-large",
        torch_dtype=torch.float16,
        attn_implementation=attn_impl
    )
    pipeline.to("cuda")
    print("✅ تم تحميل خط الأنابيب.")
except Exception as e:
    print(f"⚠️ خطأ أثناء التحميل: {e}")
    print("محاولة التحميل بدون تحديد attn_implementation...")
    pipeline = TrellisImageTo3DPipeline.from_pretrained(
        "JeffreyXiang/TRELLIS-image-large",
        torch_dtype=torch.float16
    )
    pipeline.to("cuda")
    print("✅ تم تحميل خط الأنابيب بالوضع الافتراضي.")

In [ ]:
from PIL import Image
import requests

url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/tiger.png"
try:
    image = Image.open(requests.get(url, stream=True).raw).convert("RGB")
    display(image)
except:
    print("⚠️ تعذر تحميل الصورة التجريبية. ارفع صورتك أو ضع رابطاً آخر.")

In [ ]:
print("⏳ جارٍ التوليد...")
try:
    outputs = pipeline.run(image, seed=42)
    gaussian = outputs.get('gaussian')
    mesh = outputs.get('mesh')
    print("🔹 تم التوليد بنجاح.")
except Exception as e:
    print(f"❌ فشل التوليد: {e}")
    gaussian, mesh = None, None

In [ ]:
from pathlib import Path
out_dir = Path("/content/outputs")
out_dir.mkdir(exist_ok=True)

if mesh:
    try:
        mesh.export(out_dir / "model.glb")
        print("✅ model.glb")
    except Exception as e:
        print(f"⚠️ فشل تصدير GLB: {e}")
else:
    print("ℹ️ لا توجد شبكة.")

if gaussian:
    try:
        if hasattr(gaussian[0], 'save_ply'):
            gaussian[0].save_ply(out_dir / "gaussian.ply")
            print("✅ gaussian.ply")
    except Exception as e:
        print(f"⚠️ فشل تصدير PLY: {e}")

In [ ]:
import viser
from google.colab.output import eval_js
from IPython.display import IFrame, display

PORT = 8080
server = viser.ViserServer(host="0.0.0.0", port=PORT)

if gaussian:
    try:
        from trellis.utils import render_utils
        render_utils.render_gaussian(server, gaussian[0])
        print("✅ تم عرض السحابة الغاوسية.")
    except ImportError:
        points = gaussian[0].xyz.detach().cpu().numpy().reshape(-1, 3)
        server.scene.add_point_cloud("/points", points=points, colors=(255,200,200), point_size=0.01)
        print("✅ تم عرض سحابة نقاط.")
else:
    print("لا توجد بيانات غاوسية.")

try:
    url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
    public_url = f"https://{url}"
    print(f"🔗 رابط المشاهدة: {public_url}")
    display(IFrame(src=public_url, width="100%", height="600px"))
except Exception as e:
    print(f"⚠️ تعذر العرض المضمن: {e}")

## لماذا اختفت مشكلة `git clone`؟
- نستخدم `pip install git+https://...` لتثبيت المكتبة مباشرة (لا حاجة لـ `git clone`).
- في حال فشلت تلك الطريقة، ننتقل لتنزيل أرشيف ZIP من GitHub وفك ضغطه.
- بهذه الطريقة لا نحتاج لأي مصادقة أو أوامر `git` التي قد تفشل في بعض الشبكات.

شغّل الخلايا بالترتيب.